In [ ]:
import os
import sys
from pathlib import Path
import nbformat
from flask import Flask, jsonify, request
from flask_cors import CORS

# Get the project root directory
# When running as notebook, Path.cwd() should be the backend directory
# So project root is parent of cwd
current_dir = Path.cwd()
if current_dir.name == "backend":
    project_root = current_dir.parent
else:
    # Fallback: assume we're in project root
    project_root = current_dir

backend_dir = project_root / "backend"
init_notebook_path = backend_dir / "lib" / "data" / "init.ipynb"

# Load and execute the init notebook
nb = nbformat.read(str(init_notebook_path), as_version=4)
namespace = {}
for cell in nb.cells:
    if cell.cell_type == "code":
        # Replace relative path with absolute path
        code = cell.source
        if "../../../data/data.h5" in code:
            data_path = project_root / "data" / "data.h5"
            # Replace the path string, handling both quoted and unquoted cases
            import re
            # Replace "path" or 'path' with the absolute path
            code = re.sub(r'["\']\.\.\/\.\.\/\.\.\/data\/data\.h5["\']', f'"{str(data_path)}"', code)
        exec(code, namespace)

# Extract functions and variables we need
create_plant = namespace.get('create_plant')
create_empID = namespace.get('create_empID')
hruuid = namespace.get('hruuid')
h5py = namespace.get('h5py')
datetime = namespace.get('datetime')

# Initialize Flask app
# Provide explicit root_path and import_name since we're executing from a notebook
import os
app = Flask('server', root_path=str(backend_dir))
CORS(app, origins=["http://localhost:4321", "http://localhost:3000"])

# HDF5 file path
data_file_path = project_root / "data" / "data.h5"


In [ ]:
def get_h5_file():
    """Open and return the HDF5 file handle"""
    return h5py.File(str(data_file_path), "a")

def create_plant_with_context(f):
    """Create a plant using the provided file handle"""
    # Set up the context that create_plant expects
    namespace['file'] = f
    namespace['plant_group'] = f.require_group("plants")
    # Execute create_plant in the updated namespace
    exec('plant = create_plant()', namespace)
    return namespace['plant']

def create_sequencer_effect(f, effect_type: str, row: int, col: int, properties: dict = None):
    """Create a sequencer effect group in sequencer_effects_properties and return its UUID"""
    effects_props_group = f.require_group("sequencer_effects_properties")
    effect_uuid = hruuid.generate()
    sequencer_effect = effects_props_group.require_group(effect_uuid)
    sequencer_effect.attrs["effect_type"] = effect_type
    sequencer_effect.attrs["sequencer_row"] = row
    sequencer_effect.attrs["sequencer_col"] = col
    sequencer_effect.attrs["timestamp"] = datetime.now().isoformat()
    
    # Store properties as attributes
    if properties:
        for key, value in properties.items():
            sequencer_effect.attrs[f"prop_{key}"] = value
    
    # Return the UUID
    return effect_uuid

def get_sequencer_grid():
    """Read sequencer dataset and return as 2x12 grid"""
    with get_h5_file() as f:
        if "sequencer" not in f.keys():
            # Create sequencer if it doesn't exist
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        # Handle 3D shape (2,12,0) or 2D shape (2,12)
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                # Empty 3D array, return empty 2x12 grid
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                # Use first slice
                grid = seq[:, :, 0].tolist()
                # Convert bytes to strings if needed, normalize empty values to ""
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else str(cell) if cell else "" for cell in row] for row in grid]
                # Normalize: ensure empty strings, not None or whitespace
                grid = [[cell.strip() if isinstance(cell, str) and cell.strip() else "" for cell in row] for row in grid]
        else:
            # 2D array
            grid = seq[:].tolist()
            # Convert bytes to strings if needed, normalize empty values to ""
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else str(cell) if cell else "" for cell in row] for row in grid]
            # Normalize: ensure empty strings, not None or whitespace
            grid = [[cell.strip() if isinstance(cell, str) and cell.strip() else "" for cell in row] for row in grid]
        return grid

def set_sequencer_grid(grid):
    """Write 2x12 grid to sequencer dataset"""
    with get_h5_file() as f:
        if "sequencer" not in f.keys():
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        # Reshape to 2D if needed
        if len(seq.shape) == 3:
            # Resize third dimension to 1 if needed
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            # Ensure all cells are strings, then encode
            encoded_grid = []
            for row in grid:
                encoded_row = []
                for cell in row:
                    if cell is None:
                        encoded_row.append(b'')
                    elif isinstance(cell, bytes):
                        encoded_row.append(cell)
                    elif isinstance(cell, str):
                        encoded_row.append(cell.encode('utf-8'))
                    else:
                        encoded_row.append(str(cell).encode('utf-8'))
                encoded_grid.append(encoded_row)
            seq[:, :, 0] = encoded_grid
        else:
            # Ensure all cells are strings, then encode
            encoded_grid = []
            for row in grid:
                encoded_row = []
                for cell in row:
                    if cell is None:
                        encoded_row.append(b'')
                    elif isinstance(cell, bytes):
                        encoded_row.append(cell)
                    elif isinstance(cell, str):
                        encoded_row.append(cell.encode('utf-8'))
                    else:
                        encoded_row.append(str(cell).encode('utf-8'))
                encoded_grid.append(encoded_row)
            seq[:] = encoded_grid
        
        # Explicitly flush to ensure data is written
        f.flush()

def get_effects_grid():
    """Read sequencer_effects dataset and return as 2x12 grid"""
    with get_h5_file() as f:
        if "sequencer_effects" not in f.keys():
            # Create sequencer_effects if it doesn't exist
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        # Handle 3D shape (2,12,0) or 2D shape (2,12)
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                # Empty 3D array, return empty 2x12 grid
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                # Use first slice
                grid = seq[:, :, 0].tolist()
                # Convert bytes to strings if needed
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        else:
            # 2D array
            grid = seq[:].tolist()
            # Convert bytes to strings if needed
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        return grid

def set_effects_grid(grid):
    """Write 2x12 grid to sequencer_effects dataset"""
    with get_h5_file() as f:
        if "sequencer_effects" not in f.keys():
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        # Reshape to 2D if needed
        if len(seq.shape) == 3:
            # Resize third dimension to 1 if needed
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            seq[:, :, 0] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]
        else:
            seq[:] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]


In [ ]:
@app.route('/api/plants', methods=['POST'])
def create_plant_endpoint():
    """Create a new plant"""
    try:
        f = get_h5_file()
        plant = create_plant_with_context(f)
        plant_id = plant.name.split('/')[-1]  # Get the plant ID from the group name
        timestamp = plant.attrs.get("added_timestamp", "")
        f.close()
        
        return jsonify({
            "id": plant_id,
            "added_timestamp": timestamp
        }), 201
    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/plants', methods=['GET'])
def list_plants():
    """List all plants"""
    try:
        plants = []
        with get_h5_file() as f:
            if "plants" in f.keys():
                plant_group = f["plants"]
                for plant_id in plant_group.keys():
                    plant = plant_group[plant_id]
                    plants.append({
                        "id": plant_id,
                        "added_timestamp": plant.attrs.get("added_timestamp", "")
                    })
        return jsonify({"plants": plants}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer', methods=['GET'])
def get_sequencer():
    """Get current sequencer state"""
    try:
        grid = get_sequencer_grid()
        return jsonify({"sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer', methods=['PUT'])
def update_sequencer():
    """Update sequencer position"""
    try:
        data = request.get_json()
        row = data.get('row')
        col = data.get('col')
        plant_id = data.get('plant_id', '')  # Empty string to clear
        
        if row is None or col is None:
            return jsonify({"error": "row and col are required"}), 400
        
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        grid = get_sequencer_grid()
        grid[row][col] = plant_id
        set_sequencer_grid(grid)
        
        return jsonify({"sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects', methods=['GET'])
def get_effects():
    """Get current effects grid state - returns effect types by looking up UUIDs from sequencer_effects_properties"""
    try:
        grid = get_effects_grid()
        # Convert UUIDs to effect types for frontend compatibility
        effect_types_grid = []
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            for row in grid:
                effect_types_row = []
                for uuid in row:
                    if uuid:
                        # Look up effect type from sequencer_effects_properties group
                        if uuid in effects_props_group.keys():
                            sequencer_effect = effects_props_group[uuid]
                            effect_type = sequencer_effect.attrs.get("effect_type", "")
                            effect_types_row.append(effect_type)
                        else:
                            # UUID not found, clear it
                            effect_types_row.append("")
                    else:
                        effect_types_row.append("")
                effect_types_grid.append(effect_types_row)
        return jsonify({"effects": effect_types_grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects', methods=['PUT'])
def update_effects():
    """Update effects grid position"""
    try:
        data = request.get_json()
        row = data.get('row')
        col = data.get('col')
        effect = data.get('effect', '')  # Empty string to clear
        properties = data.get('properties', {})  # Optional properties
        
        if row is None or col is None:
            return jsonify({"error": "row and col are required"}), 400
        
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        # Validate effect type
        valid_effects = ['AC', 'DC', 'AMF', 'CMF', '']
        if effect not in valid_effects:
            return jsonify({"error": f"effect must be one of {valid_effects}"}), 400
        
        grid = get_effects_grid()
        
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            
            # If clearing effect, delete the sequencer effect group and clear the grid position
            if not effect:
                old_uuid = grid[row][col]
                if old_uuid and old_uuid in effects_props_group.keys():
                    del effects_props_group[old_uuid]
                grid[row][col] = ""
            else:
                # Check if there's already an effect at this position
                old_uuid = grid[row][col]
                
                # If there's an existing effect, update it instead of creating new
                if old_uuid and old_uuid in effects_props_group.keys():
                    sequencer_effect = effects_props_group[old_uuid]
                    # Update effect type and properties
                    sequencer_effect.attrs["effect_type"] = effect
                    sequencer_effect.attrs["sequencer_row"] = row
                    sequencer_effect.attrs["sequencer_col"] = col
                    sequencer_effect.attrs["timestamp"] = datetime.now().isoformat()
                    
                    # Update properties
                    if properties:
                        for key, value in properties.items():
                            sequencer_effect.attrs[f"prop_{key}"] = value
                    
                    # Keep the same UUID
                    grid[row][col] = old_uuid
                else:
                    # Create new sequencer effect group
                    uuid = create_sequencer_effect(f, effect, row, col, properties)
                    grid[row][col] = uuid
        
        set_effects_grid(grid)
        
        # Return effect types for frontend compatibility
        effect_types_grid = []
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            for grid_row in grid:
                effect_types_row = []
                for uuid in grid_row:
                    if uuid:
                        if uuid in effects_props_group.keys():
                            sequencer_effect = effects_props_group[uuid]
                            effect_type = sequencer_effect.attrs.get("effect_type", "")
                            effect_types_row.append(effect_type)
                        else:
                            effect_types_row.append("")
                    else:
                        effect_types_row.append("")
                effect_types_grid.append(effect_types_row)
        
        return jsonify({"effects": effect_types_grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects/<int:row>/<int:col>/properties', methods=['GET'])
def get_effect_properties(row, col):
    """Get properties for an effect at a specific position"""
    try:
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        with get_h5_file() as f:
            grid = get_effects_grid()
            uuid = grid[row][col]
            
            if not uuid:
                return jsonify({"properties": {}}), 200
            
            effects_props_group = f.require_group("sequencer_effects_properties")
            if uuid not in effects_props_group.keys():
                return jsonify({"properties": {}}), 200
            
            sequencer_effect = effects_props_group[uuid]
            properties = {}
            
            # Extract all prop_* attributes
            for key in sequencer_effect.attrs.keys():
                if key.startswith("prop_"):
                    prop_name = key[5:]  # Remove "prop_" prefix
                    properties[prop_name] = sequencer_effect.attrs[key]
            
            return jsonify({
                "effect_type": sequencer_effect.attrs.get("effect_type", ""),
                "properties": properties
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects/<int:row>/<int:col>/properties', methods=['PUT'])
def update_effect_properties(row, col):
    """Update properties for an effect at a specific position"""
    try:
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        data = request.get_json()
        properties = data.get('properties', {})
        
        with get_h5_file() as f:
            grid = get_effects_grid()
            uuid = grid[row][col]
            
            if not uuid:
                return jsonify({"error": "Effect not found at this position"}), 404
            
            effects_props_group = f.require_group("sequencer_effects_properties")
            if uuid not in effects_props_group.keys():
                return jsonify({"error": "Effect not found at this position"}), 404
            
            sequencer_effect = effects_props_group[uuid]
            
            # Update properties
            for key, value in properties.items():
                sequencer_effect.attrs[f"prop_{key}"] = value
            
            # Return updated properties
            updated_properties = {}
            for key in sequencer_effect.attrs.keys():
                if key.startswith("prop_"):
                    prop_name = key[5:]
                    updated_properties[prop_name] = sequencer_effect.attrs[key]
            
            return jsonify({
                "effect_type": sequencer_effect.attrs.get("effect_type", ""),
                "properties": updated_properties
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


In [ ]:
import threading
import time
import math

# Tile allocation constants based on paper
# First 8 columns (0-7): Storage/Observation area  
# Last 4 columns (8-11): EM exposure zones (swap space available)
STORAGE_COLS = list(range(0, 8))  # Columns 0-7 for storage
EM_EXPOSURE_COLS = list(range(8, 12))  # Columns 8-11 for EM exposure

# Robot configuration: 1 robot per row (2 total)
# Robot 0 serves row 0, Robot 1 serves row 1
ROBOT_SPEED = 0.5  # columns per tick (simulated speed)
OBSERVATION_TICKS = 3  # ticks to observe a plant
PICKUP_TICKS = 2  # ticks to pick up a plant
PUTDOWN_TICKS = 2  # ticks to put down a plant
OBSERVATION_COOLDOWN_TICKS = 30  # ticks before a plant can be observed again

# Simulation state (in-memory for now)
simulation_state = {
    'running': False,
    'tick': 0,
    'speed': 1.0,  # ticks per second
    'robots': [
        {'id': 0, 'row': 0, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
        {'id': 1, 'row': 1, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0}
    ],
    'observation_queue': [],  # Plants queued for observation
    'observation_station': {'row': -1, 'col': 11},  # Camera position at rightmost column (physical limitation)
    'plants_observed': [],  # Plants that have been observed this cycle
    'last_tick_time': None
}

simulation_lock = threading.Lock()
simulation_thread = None

def get_tile_allocation():
    """Return tile allocation info for frontend"""
    return {
        'storage_cols': STORAGE_COLS,
        'em_exposure_cols': EM_EXPOSURE_COLS,
        'observation_station': simulation_state['observation_station']
    }

def calculate_travel_time(from_col, to_col):
    """Calculate ticks needed to travel between columns"""
    distance = abs(to_col - from_col)
    return math.ceil(distance / ROBOT_SPEED)

def find_available_robot(row):
    """Find an idle robot for the given row"""
    for robot in simulation_state['robots']:
        if robot['row'] == row and robot['state'] == 'idle':
            return robot
    return None

def update_robot(robot):
    """Update a single robot's state for one tick"""
    if robot['state'] == 'idle':
        return
    
    if robot['state'] == 'moving_to_pickup':
        # Move toward target
        if robot['target_col'] is not None:
            if robot['col'] < robot['target_col']:
                robot['col'] = min(robot['col'] + ROBOT_SPEED, robot['target_col'])
            elif robot['col'] > robot['target_col']:
                robot['col'] = max(robot['col'] - ROBOT_SPEED, robot['target_col'])
            
            if robot['col'] == robot['target_col']:
                robot['state'] = 'picking_up'
                robot['ticks_remaining'] = PICKUP_TICKS
    
    elif robot['state'] == 'picking_up':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Pick up complete - get plant from grid
            grid = get_sequencer_grid()
            col = int(robot['target_col'] + 0.5)  # Consistent rounding
            pickup_success = False
            if 0 <= col < 12 and grid[robot['row']][col]:
                robot['holding_plant'] = grid[robot['row']][col]
                grid[robot['row']][col] = ''
                set_sequencer_grid(grid)
                pickup_success = True
            
            if not pickup_success:
                # Pickup failed (no plant at location) - go back to idle
                robot['state'] = 'idle'
                robot['target_col'] = None
                robot['task_type'] = None
                robot['em_target_col'] = None
                return
            
            # Check task type: EM exposure vs observation
            if robot.get('task_type') == 'em_exposure':
                # Move to EM zone to place plant
                robot['state'] = 'moving_to_em_zone'
                robot['target_col'] = robot.get('em_target_col', EM_EXPOSURE_COLS[0])
            else:
                # Normal observation task
                robot['state'] = 'moving_to_observe'
                robot['target_col'] = simulation_state['observation_station']['col']
    
    elif robot['state'] == 'moving_to_em_zone':
        # Moving plant to EM zone for exposure
        target = robot['target_col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'placing_in_em_zone'
            robot['ticks_remaining'] = PUTDOWN_TICKS
    
    elif robot['state'] == 'placing_in_em_zone':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Place plant in EM zone
            if robot['holding_plant']:
                grid = get_sequencer_grid()
                col = int(robot['target_col'] + 0.5)
                placed = False
                
                if 0 <= col < 12 and not grid[robot['row']][col]:
                    grid[robot['row']][col] = robot['holding_plant']
                    set_sequencer_grid(grid)
                    
                    # Track EM exposure start
                    em_zone_exposure[robot['holding_plant']] = {
                        'tick_placed': simulation_state['tick'],
                        'row': robot['row'],
                        'col': col,
                        'original_storage_col': int(robot.get('original_col', 0))
                    }
                    placed = True
                    robot['holding_plant'] = None
                else:
                    # EM zone slot occupied - find alternative EM slot or return to storage
                    import sys
                    print(f"[WARNING] Robot {robot['id']} cannot place plant at EM zone R{robot['row']}C{col} - slot occupied", file=sys.stderr)
                    
                    # Try other EM slots in same row
                    for alt_col in EM_EXPOSURE_COLS:
                        if alt_col != col and not grid[robot['row']][alt_col]:
                            grid[robot['row']][alt_col] = robot['holding_plant']
                            set_sequencer_grid(grid)
                            em_zone_exposure[robot['holding_plant']] = {
                                'tick_placed': simulation_state['tick'],
                                'row': robot['row'],
                                'col': alt_col,
                                'original_storage_col': int(robot.get('original_col', 0))
                            }
                            placed = True
                            robot['holding_plant'] = None
                            print(f"[INFO] Robot {robot['id']} placed plant at alternative EM slot R{robot['row']}C{alt_col}", file=sys.stderr)
                            break
                    
                    if not placed:
                        # No EM slots available - return to storage instead
                        storage_col = robot.get('original_col', 0)
                        if 0 <= storage_col < 8 and not grid[robot['row']][storage_col]:
                            grid[robot['row']][storage_col] = robot['holding_plant']
                            set_sequencer_grid(grid)
                            placed = True
                            robot['holding_plant'] = None
                            print(f"[INFO] Robot {robot['id']} returned plant to storage R{robot['row']}C{storage_col} (EM zone full)", file=sys.stderr)
                        else:
                            # Try any empty storage slot
                            for alt_col in STORAGE_COLS:
                                if not grid[robot['row']][alt_col]:
                                    grid[robot['row']][alt_col] = robot['holding_plant']
                                    set_sequencer_grid(grid)
                                    placed = True
                                    robot['holding_plant'] = None
                                    print(f"[INFO] Robot {robot['id']} placed plant at storage R{robot['row']}C{alt_col} (EM zone full)", file=sys.stderr)
                                    break
                    
                    if not placed:
                        # CRITICAL: No space anywhere - keep holding and retry later
                        print(f"[CRITICAL] Robot {robot['id']} cannot place plant anywhere - keeping plant!", file=sys.stderr)
                        robot['state'] = 'idle'  # Go idle but keep holding
                        robot['target_col'] = None
                        robot['task_type'] = None
                        robot['em_target_col'] = None
                        return  # Don't clear holding_plant
            
            # Reset task state and go idle (only if plant was placed)
            if not robot['holding_plant']:  # Only reset if plant was successfully placed
                robot['task_type'] = None
                robot['em_target_col'] = None
                robot['state'] = 'idle'
                robot['target_col'] = None
    
    elif robot['state'] == 'moving_to_observe':
        target = simulation_state['observation_station']['col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'observing'
            robot['ticks_remaining'] = OBSERVATION_TICKS
    
    elif robot['state'] == 'observing':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Observation complete - process plant growth
            if robot['holding_plant']:
                simulation_state['plants_observed'].append({
                    'plant_id': robot['holding_plant'],
                    'tick': simulation_state['tick'],
                    'robot_id': robot['id']
                })
                # Remove from EM exposure tracking (observation complete)
                if robot['holding_plant'] in em_zone_exposure:
                    del em_zone_exposure[robot['holding_plant']]
                    
            robot['state'] = 'moving_to_return'
            # Return to storage position (not EM zone) so the cycle can repeat
            storage_col = robot.get('storage_return_col', robot.get('original_col', 0))
            robot['target_col'] = float(storage_col)
    
    elif robot['state'] == 'moving_to_return':
        target = robot['target_col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'putting_down'
            robot['ticks_remaining'] = PUTDOWN_TICKS
    
    elif robot['state'] == 'putting_down':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Put down complete - return plant to grid
            if robot['holding_plant']:
                grid = get_sequencer_grid()
                col = int(robot['target_col'] + 0.5)
                placed = False
                
                # Try original spot first
                if 0 <= col < 12 and not grid[robot['row']][col]:
                    grid[robot['row']][col] = robot['holding_plant']
                    placed = True
                else:
                    # Original spot occupied - find ANY empty storage slot
                    for alt_col in STORAGE_COLS:
                        if not grid[robot['row']][alt_col]:
                            grid[robot['row']][alt_col] = robot['holding_plant']
                            placed = True
                            break
                    
                    # If no storage slots, try EM zone slots
                    if not placed:
                        for alt_col in EM_EXPOSURE_COLS:
                            if not grid[robot['row']][alt_col]:
                                grid[robot['row']][alt_col] = robot['holding_plant']
                                placed = True
                                break
                
                if placed:
                    set_sequencer_grid(grid)
                else:
                    # CRITICAL: No space anywhere - keep holding and retry later
                    import sys
                    print(f"[WARNING] Robot {robot['id']} cannot place plant - no empty slots!", file=sys.stderr)
                    robot['state'] = 'idle'  # Go idle but keep holding
                    robot['target_col'] = None
                    return  # Don't clear holding_plant
                
                robot['holding_plant'] = None
            robot['state'] = 'idle'
            robot['target_col'] = None

# Track plants that have been placed in EM zone and their exposure start tick
em_zone_exposure = {}  # {plant_id: {'tick_placed': int, 'row': int, 'col': int}}
EM_EXPOSURE_TICKS = 20  # How long plants need to be in EM zone before observation

def init_em_zone_exposure_from_grid():
    """Initialize em_zone_exposure from current grid state on startup.
    Assumes any plants already in EM zone have been exposed long enough."""
    global em_zone_exposure
    grid = get_sequencer_grid()
    for row in range(2):
        for col in EM_EXPOSURE_COLS:
            plant_id = grid[row][col]
            if plant_id and plant_id not in em_zone_exposure:
                # Plant is in EM zone but not tracked - assume it's been there long enough
                em_zone_exposure[plant_id] = {
                    'tick_placed': 0,  # Will be considered fully exposed (tick 0)
                    'row': row,
                    'col': col,
                    'original_storage_col': 0  # Default to col 0 for return
                }
    if em_zone_exposure:
        import sys
        print(f"[INIT] Found {len(em_zone_exposure)} plants already in EM zone", file=sys.stderr)

# Initialize on module load
init_em_zone_exposure_from_grid()

def assign_observation_task():
    """Try to assign observation tasks to idle robots
    
    Priority:
    1. Observe plants in EM zone that have been exposed long enough
    2. Move plants from storage to EM zone for exposure
    """
    grid = get_sequencer_grid()
    effects_grid = get_effects_grid()
    
    for row in range(2):
        robot = find_available_robot(row)
        if not robot:
            continue
        
        current_tick = simulation_state['tick']
        
        # Build map of plant_id -> last observation tick for cooldown check
        last_obs_tick = {}
        for obs in simulation_state['plants_observed']:
            last_obs_tick[obs['plant_id']] = obs['tick']
        
        # PRIORITY 1: Find plants in EM zone that have been exposed long enough
        for col in EM_EXPOSURE_COLS:
            plant_id = grid[row][col]
            # Check cooldown: plant is observable if never observed OR cooldown has passed
            can_observe = plant_id and (
                plant_id not in last_obs_tick or 
                current_tick - last_obs_tick[plant_id] >= OBSERVATION_COOLDOWN_TICKS
            )
            if can_observe:
                # Check if this plant has been exposed long enough
                exposure_info = em_zone_exposure.get(plant_id)
                if exposure_info and (current_tick - exposure_info['tick_placed']) >= EM_EXPOSURE_TICKS:
                    # This plant is ready for observation!
                    robot['state'] = 'moving_to_pickup'
                    robot['target_col'] = float(col)
                    robot['original_col'] = float(col)  # EM zone col (for growth data)
                    robot['storage_return_col'] = exposure_info.get('original_storage_col', col)  # Where to return
                    return  # Only assign one task per call
        
        # PRIORITY 2: Move plants from storage to EM zone
        # Find empty spot in EM zone with an effect
        target_em_col = None
        for col in EM_EXPOSURE_COLS:
            if not grid[row][col] and effects_grid[row][col]:  # Empty spot with EM effect
                target_em_col = col
                break
        
        if target_em_col is not None:
            # Find a plant in storage to move
            for col in STORAGE_COLS:
                plant_id = grid[row][col]
                if plant_id and plant_id not in em_zone_exposure:  # Plant not already being exposed
                    # Assign task to move plant to EM zone
                    robot['state'] = 'moving_to_pickup'
                    robot['target_col'] = float(col)
                    robot['original_col'] = float(col)  # Storage col
                    robot['em_target_col'] = float(target_em_col)  # Where to place for EM exposure
                    robot['task_type'] = 'em_exposure'  # Special task type
                    return

def simulation_tick():
    """Execute one simulation tick"""
    with simulation_lock:
        if not simulation_state['running']:
            return
        
        simulation_state['tick'] += 1
        simulation_state['last_tick_time'] = time.time()
        
        # Update all robots
        for robot in simulation_state['robots']:
            update_robot(robot)
        
        # Process growth data for newly observed plants
        # Check robots that just finished observing
        for robot in simulation_state['robots']:
            if robot['state'] == 'moving_to_return' and robot['holding_plant']:
                # Check if this plant was just observed
                recent_obs = [p for p in simulation_state['plants_observed'] 
                             if p['tick'] == simulation_state['tick'] and p['plant_id'] == robot['holding_plant']]
                if recent_obs:
                    # Process growth data (fake pipeline)
                    try:
                        # Get EM effect for this plant from its ORIGINAL position
                        # (plant is being held by robot, not in grid anymore)
                        effects_grid = get_effects_grid()
                        effect_type = None
                        effect_properties = {}
                        
                        # Use the robot's original_col (where plant was picked up from)
                        original_col = int(robot.get('original_col', 0) + 0.5)
                        row = robot['row']
                        
                        # Check if original position was in EM zone
                        if original_col in EM_EXPOSURE_COLS:
                            uuid = effects_grid[row][original_col]
                            if uuid:
                                f = get_h5_file()
                                try:
                                    effects_props_group = f.require_group("sequencer_effects_properties")
                                    if uuid in effects_props_group.keys():
                                        sequencer_effect = effects_props_group[uuid]
                                        effect_type = sequencer_effect.attrs.get("effect_type", "")
                                        
                                        # Extract properties
                                        for key in sequencer_effect.attrs.keys():
                                            if key.startswith("prop_"):
                                                prop_name = key[5:]
                                                effect_properties[prop_name] = sequencer_effect.attrs[key]
                                finally:
                                    f.close()
                        
                        # Calculate and record growth
                        growth = calculate_plant_growth(robot['holding_plant'], simulation_state['tick'], effect_type, effect_properties)
                        record_growth_data(robot['holding_plant'], simulation_state['tick'], growth, effect_type, effect_properties)
                    except Exception as e:
                        import traceback
                        print(f"Error in growth pipeline: {traceback.format_exc()}")
        
        # Try to assign new tasks
        assign_observation_task()
        
        # Auto-update test queue and assign queued tests to empty EM zones (every 25 ticks for more responsive updates)
        if simulation_state['tick'] % 25 == 0:
            try:
                import sys
                
                # Update queue based on current data
                queue_size_before = len(test_queue)
                update_test_queue()
                queue_size_after = len(test_queue)
                print(f"[QUEUE] Updated: {queue_size_before} -> {queue_size_after} configs at tick {simulation_state['tick']}", file=sys.stderr)
                
                # Auto-assign queued tests to empty EM zones
                grid = get_effects_grid()
                queue_index = 0
                assigned_count = 0
                
                for row in range(2):
                    for col in EM_EXPOSURE_COLS:
                        uuid = grid[row][col]
                        has_effect = False
                        
                        if uuid:
                            with get_h5_file() as f:
                                effects_props_group = f.require_group("sequencer_effects_properties")
                                if uuid in effects_props_group.keys():
                                    has_effect = True
                        
                        # Check if we should replace existing effect or assign to empty slot
                        should_replace = False
                        if has_effect and len(test_queue) > 0:
                            # More aggressive: Replace if current effect has been tested >= 3 times
                            # OR if queue has high-priority configs (priority > 1000)
                            with get_h5_file() as f:
                                effects_props_group = f.require_group("sequencer_effects_properties")
                                if uuid in effects_props_group.keys():
                                    sequencer_effect = effects_props_group[uuid]
                                    effect_type_curr = sequencer_effect.attrs.get("effect_type", "")
                                    
                                    # Extract properties
                                    properties_curr = {}
                                    for key in sequencer_effect.attrs.keys():
                                        if key.startswith("prop_"):
                                            prop_name = key[5:]
                                            properties_curr[prop_name] = sequencer_effect.attrs[key]
                                    
                                    # Check test count
                                    test_count_curr = get_config_test_count(effect_type_curr, properties_curr)
                                    
                                    # Replace if: tested >= MAX_RETESTS OR tested >= 3 times (more aggressive)
                                    if test_count_curr >= MAX_RETESTS or test_count_curr >= 3:
                                        should_replace = True
                                    # Also replace if queue has high-priority items waiting
                                    elif len(test_queue) > queue_index:
                                        next_item = test_queue[queue_index] if queue_index < len(test_queue) else None
                                        if next_item and next_item.get('priority', 0) > 1000:
                                            should_replace = True  # High priority config waiting
                        
                        # Assign new config if empty OR if should replace
                        if not has_effect or should_replace:
                            # Find next valid config from queue
                            config_assigned = False
                            while queue_index < len(test_queue):
                                queue_item = test_queue[queue_index]
                                effect_type, properties = queue_item['config']
                                
                                # Check if can still test this config
                                test_count = get_config_test_count(effect_type, properties)
                                if test_count < MAX_RETESTS:
                                    f = get_h5_file()
                                    try:
                                        # Clear old effect if replacing
                                        if should_replace and uuid:
                                            effects_props_group = f.require_group("sequencer_effects_properties")
                                            if uuid in effects_props_group.keys():
                                                del effects_props_group[uuid]
                                        
                                        new_uuid = create_sequencer_effect(f, effect_type, row, col, properties)
                                        grid[row][col] = new_uuid
                                        assigned_count += 1
                                        props_str = ', '.join([f"{k}={v}" for k, v in properties.items()])
                                        action = "Replaced" if should_replace else "Assigned"
                                        print(f"[QUEUE] {action} {effect_type} {props_str} to R{row}C{col} (test_count: {test_count})", file=sys.stderr)
                                        queue_index += 1
                                        config_assigned = True
                                        break
                                    finally:
                                        f.close()
                                else:
                                    queue_index += 1  # Skip this one (reached max retests)
                            
                            if not config_assigned and not has_effect:
                                # Queue exhausted or all configs reached max retests - only for empty slots
                                # Fall back to intelligent config
                                config = get_intelligent_em_config(row, col)
                                if config:
                                    effect_type, properties = config
                                    f = get_h5_file()
                                    try:
                                        new_uuid = create_sequencer_effect(f, effect_type, row, col, properties)
                                        grid[row][col] = new_uuid
                                        assigned_count += 1
                                        print(f"[QUEUE] Fallback: Assigned {effect_type} to R{row}C{col} (queue exhausted)", file=sys.stderr)
                                    finally:
                                        f.close()
                
                if assigned_count > 0:
                    set_effects_grid(grid)
                    print(f"[QUEUE] Assigned {assigned_count} new effects to EM zones", file=sys.stderr)
            except Exception as e:
                import sys
                import traceback
                print(f"[QUEUE] Error in auto-assignment: {e}", file=sys.stderr)
                traceback.print_exc(file=sys.stderr)

# Minimum sleep interval to prevent excessive CPU usage at high speeds
MIN_SLEEP_INTERVAL = 0.001  # 1ms minimum for ultra-fast debugging

def simulation_loop():
    """Main simulation loop running in background thread"""
    while True:
        with simulation_lock:
            if not simulation_state['running']:
                break
            speed = simulation_state['speed']
        
        simulation_tick()
        # Calculate sleep with minimum threshold to prevent high CPU usage
        sleep_time = max(MIN_SLEEP_INTERVAL, 1.0 / speed if speed > 0 else 1.0)
        time.sleep(sleep_time)

def start_simulation():
    """Start the simulation"""
    global simulation_thread
    with simulation_lock:
        if simulation_state['running']:
            return False
        simulation_state['running'] = True
    
    simulation_thread = threading.Thread(target=simulation_loop, daemon=True)
    simulation_thread.start()
    return True

def stop_simulation():
    """Stop the simulation"""
    with simulation_lock:
        simulation_state['running'] = False
    return True

def reset_simulation():
    """Reset simulation state"""
    with simulation_lock:
        simulation_state['running'] = False
        simulation_state['tick'] = 0
        simulation_state['speed'] = 1.0
        simulation_state['robots'] = [
            {'id': 0, 'row': 0, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 1, 'row': 0, 'col': 3.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 2, 'row': 1, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 3, 'row': 1, 'col': 3.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0}
        ]
        simulation_state['observation_queue'] = []
        simulation_state['plants_observed'] = []
        simulation_state['last_tick_time'] = None
    return True


In [ ]:
@app.route('/api/clear-all', methods=['POST'])
def clear_all_data():
    """Clear all sequencer data, effects, growth data, and reset simulation"""
    import sys
    try:
        # Stop simulation first
        with simulation_lock:
            simulation_state['running'] = False

        # Clear sequencer grid (plants)
        grid = get_sequencer_grid()
        for row in range(2):
            for col in range(12):
                grid[row][col] = ""
        set_sequencer_grid(grid)

        # Clear effects grid
        effects_grid = get_effects_grid()
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            # Delete all effect groups
            effect_uuids = list(effects_props_group.keys())
            for uuid in effect_uuids:
                del effects_props_group[uuid]

            # Clear effects grid
            for row in range(2):
                for col in range(12):
                    effects_grid[row][col] = ""
            set_effects_grid(effects_grid)

        # Clear all plant growth data
        with get_h5_file() as f:
            if "plant_growth_data" in f:
                del f["plant_growth_data"]
                print("[CLEAR-ALL] Deleted plant_growth_data group", file=sys.stderr)

        # Clear in-memory growth data (restore defaultdict to avoid KeyErrors)
        global plant_growth_data, _recorded_observations
        from collections import defaultdict
        plant_growth_data = defaultdict(lambda: {'ticks': [], 'growth': [], 'em_effect': None, 'em_properties': {}})
        _recorded_observations = set()

        # Reset points system
        global points_system
        points_system = {
            'data_points': {},
            'target_coverage': 10,
            'exploration_bonus': 5
        }

        # Clear test queue
        global test_queue
        test_queue = []

        # Reset em_zone_exposure
        global em_zone_exposure
        em_zone_exposure = {}

        # Reset simulation state
        reset_simulation()

        print("[CLEAR-ALL] All data cleared successfully", file=sys.stderr)

        return jsonify({
            'success': True,
            'message': 'All sequencer data, effects, growth data, and simulation state cleared'
        }), 200
    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"[CLEAR-ALL ERROR] {error_msg}", file=sys.stderr)
        return jsonify({"error": str(e), "traceback": error_msg}), 500

In [ ]:
import random
from collections import defaultdict

# Plant growth data storage (in-memory for now, can be moved to HDF5 later)
plant_growth_data = defaultdict(lambda: {'ticks': [], 'growth': [], 'em_effect': None, 'em_properties': {}})

# Points system for intelligent EM effect selection
points_system = {
    'data_points': {},  # {(effect_type, properties_hash): count}
    'target_coverage': 10,  # Target number of data points per unique EM configuration
    'exploration_bonus': 5  # Bonus points for exploring new configurations
}

# Test queue system for intelligent EM effect generation
test_queue = []  # List of {config, priority, test_count, reason}
MAX_RETESTS = 8  # Maximum times a configuration can be retested

def hash_em_properties(effect_type, properties):
    """Create a hash for EM effect properties"""
    props_str = f"{effect_type}:{sorted(properties.items())}"
    return hash(props_str)

def get_config_test_count(effect_type, properties):
    """Get how many times a config has been tested"""
    config_hash = hash_em_properties(effect_type, properties)
    return points_system['data_points'].get(config_hash, 0)

def calculate_plant_growth(plant_id, tick, effect_type=None, effect_properties=None):
    """Calculate growth value based on effect; placeholder model with variability"""
    base_growth = 1.0
    # Simple diversified multiplier based on effect type
    if not effect_type or effect_type == 'Control':
        effect_multiplier = 1.0 + random.gauss(0, 0.05)
        std_dev = 0.05
    elif effect_type == 'AC':
        effect_multiplier = 1.05 + random.gauss(0, 0.07)
        std_dev = 0.07
    elif effect_type == 'DC':
        effect_multiplier = 0.95 + random.gauss(0, 0.06)
        std_dev = 0.06
    elif effect_type == 'AMF':
        effect_multiplier = 1.02 + random.gauss(0, 0.08)
        std_dev = 0.08
    else:  # CMF and others
        effect_multiplier = 1.10 + random.gauss(0, 0.11)
        std_dev = 0.11

    growth = base_growth * effect_multiplier
    growth = max(0.5, min(2.0, growth))
    return growth

def get_points_for_config(effect_type, properties):
    """Calculate points for a given EM configuration"""
    config_hash = hash_em_properties(effect_type, properties)
    count = points_system['data_points'].get(config_hash, 0)
    if count == 0:
        return points_system['exploration_bonus']
    else:
        return max(0, points_system['target_coverage'] - count)

# Track which plant-tick combinations have been recorded to prevent duplicates
_recorded_observations = set()

# Unified EM parameter schema - all growth points have ALL EM parameters
# Parameters set to 0 when that effect type is not applied
def create_unified_em_params(effect_type=None, effect_properties=None):
    """Create a unified EM parameters dict with all parameters, zeroing non-applied ones"""
    params = {
        # AC parameters
        'ac_frequency': 0.0,
        'ac_voltage': 0.0,
        'ac_phase': 0.0,
        # DC parameters
        'dc_voltage': 0.0,
        'dc_current': 0.0,
        # AMF parameters
        'amf_frequency': 0.0,
        'amf_amplitude': 0.0,
        'amf_phase': 0.0,
        # CMF parameters
        'cmf_strength': 0.0,
        'cmf_direction': 0.0,  # 0=none, 1=vertical, 2=horizontal
        # Effect type indicator (for filtering)
        'effect_type': effect_type or 'Control'
    }

    if effect_type and effect_properties:
        if effect_type == 'AC':
            params['ac_frequency'] = float(effect_properties.get('frequency', 60))
            params['ac_voltage'] = float(effect_properties.get('voltage', 120))
            params['ac_phase'] = float(effect_properties.get('phase', 0))
        elif effect_type == 'DC':
            params['dc_voltage'] = float(effect_properties.get('voltage', 12))
            params['dc_current'] = float(effect_properties.get('current', 1))
        elif effect_type == 'AMF':
            params['amf_frequency'] = float(effect_properties.get('frequency', 50))
            params['amf_amplitude'] = float(effect_properties.get('amplitude', 0.1))
            params['amf_phase'] = float(effect_properties.get('phase', 0))
        elif effect_type == 'CMF':
            params['cmf_strength'] = float(effect_properties.get('strength', 0.5))
            direction = effect_properties.get('direction', 'vertical')
            params['cmf_direction'] = 1.0 if direction == 'vertical' else 2.0 if direction == 'horizontal' else 0.0

    return params

def record_growth_data(plant_id, tick, growth, effect_type=None, effect_properties=None):
    """Record plant growth data with unified EM schema - stores both in-memory and to h5 file"""
    import sys

    # Prevent duplicate recordings for the same plant-tick combination
    obs_key = (plant_id, tick)
    if obs_key in _recorded_observations:
        print(f"[GROWTH] Skipping duplicate: {plant_id[:20]}... at tick {tick}", file=sys.stderr)
        return
    _recorded_observations.add(obs_key)

    # Ensure per-plant structure exists even if defaultdict was reset
    if plant_id not in plant_growth_data:
        plant_growth_data[plant_id] = {'ticks': [], 'growth': [], 'em_effect': None, 'em_properties': {}}

    # Create unified EM parameters
    em_params = create_unified_em_params(effect_type, effect_properties)

    print(f"[GROWTH] Recording: {plant_id[:20]}... tick={tick} growth={growth:.3f} em={effect_type}", file=sys.stderr)

    # Store in memory for quick access (using new unified schema)
    if 'unified_data' not in plant_growth_data[plant_id]:
        plant_growth_data[plant_id]['unified_data'] = []

    plant_growth_data[plant_id]['unified_data'].append({
        'tick': tick,
        'growth': growth,
        **em_params
    })

    # Also maintain legacy format for backward compatibility
    plant_growth_data[plant_id]['ticks'].append(tick)
    plant_growth_data[plant_id]['growth'].append(growth)
    if effect_type:
        plant_growth_data[plant_id]['em_effect'] = effect_type
        plant_growth_data[plant_id]['em_properties'] = effect_properties.copy() if effect_properties else {}

        # Update points system
        config_hash = hash_em_properties(effect_type, effect_properties or {})
        points_system['data_points'][config_hash] = points_system['data_points'].get(config_hash, 0) + 1

    # Persist to h5 file with unified schema
    try:
        f = h5py.File(str(data_file_path), "a")
        try:
            growth_group = f.require_group("plant_growth_data")
            plant_group = growth_group.require_group(plant_id)

            # Store using unified dataset columns
            em_param_keys = ['ac_frequency', 'ac_voltage', 'ac_phase', 'dc_voltage', 'dc_current',
                           'amf_frequency', 'amf_amplitude', 'amf_phase', 'cmf_strength', 'cmf_direction']

            if "ticks" not in plant_group:
                # Create all datasets
                plant_group.create_dataset("ticks", data=[tick], maxshape=(None,), dtype='i4')
                plant_group.create_dataset("growth", data=[growth], maxshape=(None,), dtype='f8')
                for key in em_param_keys:
                    plant_group.create_dataset(key, data=[em_params[key]], maxshape=(None,), dtype='f8')
            else:
                # Resize and append to all datasets
                new_size = plant_group["ticks"].shape[0] + 1
                plant_group["ticks"].resize((new_size,))
                plant_group["growth"].resize((new_size,))
                plant_group["ticks"][-1] = tick
                plant_group["growth"][-1] = growth

                for key in em_param_keys:
                    if key not in plant_group:
                        # Create missing dataset with zeros
                        plant_group.create_dataset(key, data=[0.0] * new_size, maxshape=(None,), dtype='f8')
                    else:
                        plant_group[key].resize((new_size,))
                    plant_group[key][-1] = em_params[key]

            # Store effect type as attribute
            plant_group.attrs["em_effect"] = effect_type or "Control"

            f.flush()
        finally:
            f.close()
    except Exception as e:
        import traceback
        print(f"[GROWTH ERROR] Persisting failed: {e}", file=sys.stderr)
        traceback.print_exc(file=sys.stderr)

# Track which config index to use next (cycles through)
_next_config_idx = 0

def get_intelligent_em_config(row, col):
    """
    Intelligently select EM effect configuration to fill the graph.
    Returns (effect_type, properties) tuple
    Cycles through different effect types and parameters for variety.
    """
    global _next_config_idx

    # Get current effects grid to see what's already set
    grid = get_effects_grid()
    current_uuid = grid[row][col]

    # If there's already an effect, return None (don't change)
    if current_uuid:
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            if current_uuid in effects_props_group.keys():
                sequencer_effect = effects_props_group[current_uuid]
                effect_type = sequencer_effect.attrs.get("effect_type", "")
                if effect_type:
                    return None

    varied_configs = [
        ('AC', {'frequency': 30, 'voltage': 60, 'phase': 0}),
        ('DC', {'voltage': 6, 'current': 0.5}),
        ('AMF', {'frequency': 50, 'amplitude': 0.1, 'phase': 0}),
        ('CMF', {'strength': 0.3, 'direction': 'vertical'}),
        ('AC', {'frequency': 100, 'voltage': 120, 'phase': 0}),
        ('DC', {'voltage': 12, 'current': 1.0}),
        ('AMF', {'frequency': 100, 'amplitude': 0.2, 'phase': 0}),
        ('CMF', {'strength': 0.7, 'direction': 'horizontal'}),
    ]

    config = varied_configs[_next_config_idx % len(varied_configs)]
    _next_config_idx += 1

    return config

# API endpoints for plant growth data

@app.route('/api/plant-growth/<plant_id>', methods=['GET'])
def get_plant_growth(plant_id):
    """Get growth data for a specific plant"""
    try:
        if plant_id not in plant_growth_data:
            return jsonify({"ticks": [], "growth": [], "em_effect": None, "em_properties": {}}), 200

        data = plant_growth_data[plant_id]
        return jsonify({
            "ticks": data['ticks'],
            "growth": data['growth'],
            "em_effect": data['em_effect'],
            "em_properties": data['em_properties']
        }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/plant-growth/all', methods=['GET'])
def get_all_plant_growth():
    """Get growth data for all plants - loads from h5 file with unified EM schema"""
    try:
        all_data = {}

        def to_native(val):
            """Convert numpy types to native Python types"""
            if hasattr(val, 'item'):
                return val.item()
            return val

        em_param_keys = ['ac_frequency', 'ac_voltage', 'ac_phase', 'dc_voltage', 'dc_current',
                        'amf_frequency', 'amf_amplitude', 'amf_phase', 'cmf_strength', 'cmf_direction']

        # Load from h5 file (persistent storage)
        with get_h5_file() as f:
            if "plant_growth_data" in f:
                growth_group = f["plant_growth_data"]
                for plant_id in growth_group.keys():
                    plant_group = growth_group[plant_id]

                    ticks = [to_native(x) for x in plant_group["ticks"][:]] if "ticks" in plant_group else []
                    growth = [to_native(x) for x in plant_group["growth"][:]] if "growth" in plant_group else []
                    em_effect = str(plant_group.attrs.get("em_effect", "Control"))

                    em_params = {}
                    for key in em_param_keys:
                        if key in plant_group:
                            em_params[key] = [to_native(x) for x in plant_group[key][:]]
                        else:
                            em_params[key] = [0.0] * len(ticks)

                    all_data[plant_id] = {
                        "ticks": ticks,
                        "growth": growth,
                        "em_effect": em_effect,
                        **em_params
                    }

        # Also include any in-memory unified data
        for plant_id, data in plant_growth_data.items():
            if plant_id not in all_data and 'unified_data' in data:
                unified = data['unified_data']
                all_data[plant_id] = {
                    "ticks": [d['tick'] for d in unified],
                    "growth": [d['growth'] for d in unified],
                    "em_effect": unified[-1].get('effect_type', 'Control') if unified else 'Control',
                    **{key: [d.get(key, 0.0) for d in unified] for key in em_param_keys}
                }

        return jsonify(all_data), 200
    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/plant-growth/unified', methods=['GET'])
def get_unified_growth_data():
    """Get all growth data as flat array of unified data points for graphing"""
    try:
        data_points = []

        def to_native(val):
            if hasattr(val, 'item'):
                return val.item()
            return val

        em_param_keys = ['ac_frequency', 'ac_voltage', 'ac_phase', 'dc_voltage', 'dc_current',
                        'amf_frequency', 'amf_amplitude', 'amf_phase', 'cmf_strength', 'cmf_direction']

        # Load from h5 file
        with get_h5_file() as f:
            if "plant_growth_data" in f:
                growth_group = f["plant_growth_data"]
                for plant_id in growth_group.keys():
                    plant_group = growth_group[plant_id]

                    ticks = plant_group["ticks"][:] if "ticks" in plant_group else []
                    growth_vals = plant_group["growth"][:] if "growth" in plant_group else []
                    em_effect = str(plant_group.attrs.get("em_effect", "Control"))

                    em_arrays = {}
                    for key in em_param_keys:
                        if key in plant_group:
                            em_arrays[key] = plant_group[key][:]
                        else:
                            em_arrays[key] = [0.0] * len(ticks)

                    for i in range(len(ticks)):
                        point = {
                            'plant_id': plant_id,
                            'tick': to_native(ticks[i]),
                            'growth': to_native(growth_vals[i]),
                        }
                        for key in em_param_keys:
                            point[key] = to_native(em_arrays[key][i]) if i < len(em_arrays[key]) else 0.0

                        if point['ac_frequency'] > 0 or point['ac_voltage'] > 0:
                            point['effect_type'] = 'AC'
                        elif point['dc_voltage'] > 0 or point['dc_current'] > 0:
                            point['effect_type'] = 'DC'
                        elif point['amf_frequency'] > 0 or point['amf_amplitude'] > 0:
                            point['effect_type'] = 'AMF'
                        elif point['cmf_strength'] > 0:
                            point['effect_type'] = 'CMF'
                        else:
                            point['effect_type'] = em_effect

                        data_points.append(point)

        # Add in-memory data
        for plant_id, data in plant_growth_data.items():
            if 'unified_data' in data:
                for d in data['unified_data']:
                    point = {'plant_id': plant_id, **d}
                    if point not in data_points:
                        data_points.append(point)

        return jsonify({
            "data_points": data_points,
            "parameters": em_param_keys + ['growth'],
            "count": len(data_points)
        }), 200
    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/sequencer/auto-populate', methods=['POST'])
def auto_populate_sequencer():
    """Auto-populate the sequencer with as many plants as possible"""
    try:
        import sys

        grid = get_sequencer_grid()
        created_plants = []
        empty_slots_found = 0

        # Fill all storage columns (0-7) for both rows
        for row in range(2):
            for col in STORAGE_COLS:
                cell_value = grid[row][col]

                is_empty = False
                if cell_value is None:
                    is_empty = True
                elif isinstance(cell_value, bytes):
                    is_empty = len(cell_value) == 0 or cell_value.decode('utf-8', errors='ignore').strip() == ""
                elif isinstance(cell_value, str):
                    is_empty = cell_value.strip() == ""
                else:
                    is_empty = not bool(cell_value)

                if is_empty:
                    empty_slots_found += 1
                    try:
                        f = get_h5_file()
                        try:
                            plant = create_plant_with_context(f)
                            plant_id = plant.name.split('/')[-1]

                            grid[row][col] = plant_id
                            created_plants.append({
                                "plant_id": plant_id,
                                "row": row,
                                "col": col
                            })
                            print(f"[AUTO-POPULATE] Created plant {plant_id[:20]}... at R{row}C{col}", file=sys.stderr)
                        finally:
                            f.close()
                    except Exception as plant_error:
                        print(f"[AUTO-POPULATE] Error creating plant at R{row}C{col}: {plant_error}", file=sys.stderr)
                        import traceback
                        traceback.print_exc(file=sys.stderr)

        print(f"[AUTO-POPULATE] Found {empty_slots_found} empty slots, created {len(created_plants)} plants", file=sys.stderr)

        set_sequencer_grid(grid)

        return jsonify({
            "success": True,
            "created_count": len(created_plants),
            "created_plants": created_plants,
            "empty_slots_found": empty_slots_found
        }), 200
    except Exception as e:
        import traceback
        import sys
        error_msg = traceback.format_exc()
        print(f"[AUTO-POPULATE ERROR] {error_msg}", file=sys.stderr)
        return jsonify({"error": str(e), "traceback": error_msg}), 500

In [ ]:
# Simulation API endpoints

@app.route('/api/simulation/state', methods=['GET'])
def get_simulation_state():
    """Get current simulation state"""
    try:
        with simulation_lock:
            return jsonify({
                'running': simulation_state['running'],
                'tick': simulation_state['tick'],
                'speed': simulation_state['speed'],
                'robots': simulation_state['robots'],
                'plants_observed': simulation_state['plants_observed'][-10:],  # Last 10
                'tile_allocation': get_tile_allocation()
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/start', methods=['POST'])
def start_simulation_endpoint():
    """Start the simulation"""
    try:
        started = start_simulation()
        return jsonify({'success': started, 'running': simulation_state['running']}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/pause', methods=['POST'])
def pause_simulation_endpoint():
    """Pause the simulation"""
    try:
        stopped = stop_simulation()
        return jsonify({'success': stopped, 'running': simulation_state['running']}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/reset', methods=['POST'])
def reset_simulation_endpoint():
    """Reset the simulation"""
    try:
        reset = reset_simulation()
        return jsonify({'success': reset, 'tick': 0}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/speed', methods=['PUT'])
def set_simulation_speed():
    """Set simulation speed (ticks per second)"""
    try:
        data = request.get_json()
        speed = data.get('speed', 1.0)
        
        # Clamp speed between 0.1 and 1000.0 (ultra-fast for debugging)
        speed = max(0.1, min(1000.0, float(speed)))
        
        with simulation_lock:
            simulation_state['speed'] = speed
        
        return jsonify({'speed': speed}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/tick', methods=['POST'])
def manual_tick():
    """Manually advance one tick (for debugging/testing)"""
    try:
        with simulation_lock:
            was_running = simulation_state['running']
            simulation_state['running'] = True
        
        simulation_tick()
        
        with simulation_lock:
            simulation_state['running'] = was_running
            return jsonify({
                'tick': simulation_state['tick'],
                'robots': simulation_state['robots']
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/tile-allocation', methods=['GET'])
def get_tile_allocation_endpoint():
    """Get tile allocation info"""
    try:
        return jsonify(get_tile_allocation()), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


In [ ]:
if __name__ == "__main__":
    print(f"Starting Flask server on http://localhost:5000")
    print(f"API endpoints:")
    print(f"  POST /api/plants - Create a new plant")
    print(f"  GET /api/plants - List all plants")
    print(f"  GET /api/sequencer - Get sequencer grid")
    print(f"  PUT /api/sequencer - Update sequencer position")
    print(f"  GET /api/effects - Get effects grid")
    print(f"  PUT /api/effects - Update effects grid position")
    print(f"  GET /api/effects/<row>/<col>/properties - Get effect properties")
    print(f"  PUT /api/effects/<row>/<col>/properties - Update effect properties")
    print(f"  GET /api/simulation/state - Get simulation state")
    print(f"  POST /api/simulation/start - Start simulation")
    print(f"  POST /api/simulation/pause - Pause simulation")
    print(f"  POST /api/simulation/reset - Reset simulation")
    print(f"  PUT /api/simulation/speed - Set simulation speed")
    print(f"  POST /api/simulation/tick - Manual tick")
    print(f"  GET /api/tile-allocation - Get tile allocation")
    app.run(host='0.0.0.0', port=5000, debug=True)
    print(f"  POST /api/clear-all - Clear all sequencer data, effects, growth data, and reset simulation")
